## Part 1 -- Example of how to generate a cluster with resolved multiple systems

In [12]:
from spisea import synthetic, evolution, atmospheres, reddening, ifmr
from spisea.imf import imf, multiplicity
import os, sys, math
import numpy as np
import matplotlib.pyplot as plt

First we must generate the isochrone which takes a few minutes if it has not been done before. This is the same as without resolved binaries

In [13]:
# Fetch isochrone
logAge = 6.70 # Age in log(years)
AKs = 1.0 # Ks filter extinction in mags
dist = 4000 # distance in parsecs
metallicity = 0 # metallicity in [M/H]
atm_func = atmospheres.get_merged_atmosphere
evo_merged = evolution.MISTv1()
redlaw = reddening.RedLawCardelli(3.1) # Rv = 3.1
filt_list = ['nirc2,J', 'nirc2,Kp']

iso_dir = 'iso_merged_r1/'

if not os.path.exists(iso_dir):
        os.mkdir(iso_dir)

iso_merged = synthetic.IsochronePhot(logAge, AKs, dist, metallicity=metallicity,
                                 evo_model=evo_merged, atm_func=atm_func,
                                 filters=filt_list, red_law=redlaw,
                                 iso_dir=iso_dir, mass_sampling=3)

Next we make the cluster specifiying the MultiplicityResolvedDK multiplicity object and the ResolvedCluster object

In [14]:
# Now we can make the cluster. 
clust_mtot = 10**3.
clust_multiplicity = multiplicity.MultiplicityResolvedDK()

# Multiplicity is defined in the IMF object
clust_imf_Mult = imf.Kroupa_2001(multiplicity=clust_multiplicity)

In [32]:
# Make clusters
clust_Mult = synthetic.ResolvedCluster(iso_merged, clust_imf_Mult, clust_mtot)

clust_Mult_ss = clust_Mult.star_systems
clust_Mult_css = clust_Mult.companions

Found 1126 stars out of mass range
Found 130 companions out of stellar mass range


In [33]:
print(dir(clust_Mult))
clust_Mult_css

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_make_companions_table', '_make_star_systems_table', '_remove_bad_systems', 'cluster_mass', 'companions', 'filt_names', 'ifmr', 'imf', 'iso', 'iso_interps', 'seed', 'set_filter_names', 'star_systems', 'verbose']


system_idx,mass,Teff,L,logg,isWR,mass_current,phase,metallicity,m_nirc2_J,m_nirc2_Kp,log_a,e,i,Omega,omega
int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
0,0.38719285100587897,3528.562055707508,5.3070253234278235e+25,4.029402859340647,0.0,0.38719237261972317,-1.0,0.0,20.558685156559683,18.296221864637108,1.8480358975402076,0.9008916536693806,123.62562640075507,142.21040966368017,249.58358835088035
2,0.2289621169520412,3266.708149954387,2.455601331385897e+25,3.999536505329441,0.0,0.22896186496749932,-1.0,0.0,21.26612104856928,19.08361653476692,0.5219457119066752,0.7317832814746495,51.96235765858219,103.61984294718714,101.67702004482612
6,0.030187780246635192,nan,nan,nan,nan,nan,nan,0.0,nan,nan,0.08011032252201078,0.43404664661496706,107.99002998566307,11.053119915751676,107.3359314023533
6,0.2367718553597917,3279.2445143878463,2.577374814382302e+25,4.000451939834175,0.0,0.23677159279693977,-1.0,0.0,21.22043431580318,19.032949951647208,0.41967615284238535,0.7352345713069193,124.00532687649682,199.3907049548153,213.8235883880024
8,0.5331083756061922,3719.2025225136326,8.35532629062078e+25,4.062223654957941,0.0,0.533107669027105,-1.0,0.0,20.14885859067925,17.861633261658625,2.311453734644365,0.412749675758433,42.6314927930793,285.31804756013844,217.96225193861534
8,0.19929946750681546,3216.574432613194,2.0320380631202656e+25,3.9950548210129253,0.0,0.19929925434082355,-1.0,0.0,21.45003492213925,19.288913435688343,-0.7299374551459901,0.9866278816927483,89.679611997554,53.59875609986605,55.07687687241135
9,0.627501014909893,3828.728183656521,1.0601901503756755e+26,4.079938793990583,0.0,0.6275001542131327,-1.0,0.0,19.93754623689446,17.64354045196632,2.9767810953183984,0.9946447659643055,117.75588245863023,106.05739290802107,78.57459292446545
10,0.9790588534827629,4280.541675224375,2.2995389250449666e+26,4.132265639566635,0.0,0.9790573747150659,-1.0,0.0,19.228409474238322,17.032769954724944,1.6327419748552827,0.6677367079013031,65.69190032869334,27.037051920600085,39.10100726081235
12,0.06485723878340402,nan,nan,nan,nan,nan,nan,0.0,nan,nan,-1.0560143495553551,0.34493184356652645,39.06650113821024,149.5747790658178,164.19399967096967


## Part 2 -- Project cluster and plot orbits of companions

In [16]:
#for now orbits.py is is https://github.com/nsabrams/Microlensing_Multiple_Systems
#clone that repo and change the following path to where the repo is located
sys.path.insert(1, '../../Microlensing_Multiple_Systems')
import orbits as binary_orbits

Now we can add the random positions of the primary stars and the calculated positions of the companions

In [18]:
print(dir(binary_orbits.add_mult_positions))

['__annotations__', '__builtins__', '__call__', '__class__', '__closure__', '__code__', '__defaults__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__get__', '__getattribute__', '__getstate__', '__globals__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__kwdefaults__', '__le__', '__lt__', '__module__', '__name__', '__ne__', '__new__', '__qualname__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__']


In [17]:
clust_Mult_ss_pos = binary_orbits.add_positions(clust_Mult_ss)
clust_Mult.companions_pos = binary_orbits.add_mult_positions(clust_Mult.companions, clust_Mult_ss_pos, logAge)

TypeError: Orbit.kep2xyz() got an unexpected keyword argument 'mass'

Next we can display the projected cluster where blue dots are the primary objects, orange are the secondary objects, and grey lines connect secondary objects with their primary one

In [10]:
binary_orbits.plot_projected_cluster(clust_Mult_ss_pos, clust_Mult.companions_pos)

AttributeError: 'ResolvedCluster' object has no attribute 'companions_pos'

Finally we can plot a random orbit with the primary object is at (0,0) of a companion where the final position is marked with a star. A specific system can be ploted by specifying system = index_number in the function

In [11]:
binary_orbits.plot_companion_orbit(clust_Mult_ss_pos, clust_Mult.companions_pos, logAge)

AttributeError: 'ResolvedCluster' object has no attribute 'companions_pos'